In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS e_earthquake_connection
TYPE HTTP
OPTIONS (
    host = 'https://earthquake.usgs.gov',
    port = 443,
    base_path = '/earthquakes/feed/v1.0/',
    bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient
w=WorkspaceClient()
conn=w.connections.get("e_earthquake_connection")
print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
dbutils.widgets.text('catalog_name','my_first_project','my_first_project')
catalog_name=dbutils.widgets.get('catalog_name')


In [0]:
spark.sql(f"use catalog {catalog_name}")
spark.sql(f"use schema e_bronze_layer")
spark.sql("create  volume if not exists e_earthquake_volume")


In [0]:
%sql
use catalog my_first_project;
use schema e_bronze_layer;
create  volume if not exists e_earthquake_volume;


In [0]:
import json
import datetime
import requests

url = f"{base_url}summary/all_day.geojson"
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Error in getting data from {url}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(
    f"/Volumes/{catalog_name}/e_bronze_layer/e_earthquake_volume/e_earthquake_volume_{current_date}.json",
    json.dumps(data),
    overwrite=True,
)